# Sentiment Analysis using SVM with BoW and TF-IDF

## CSE 4221 — Natural Language Processing Assignment

**Model:** Support Vector Machine (Linear)  
**Features:** Bag of Words (300) + TF-IDF (300)  
**Dataset:** IMDB Movie Review Dataset  
**Task:** Binary Sentiment Classification

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re, warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
print('All libraries imported successfully.')

: 

---
## 2. Load Dataset

In [ ]:
import gdown
url = 'https://drive.google.com/uc?id=1GOOxQZSmtQZ4q-UPVh87DHOgI79cx9vI'
gdown.download(url, 'IMDB-Dataset.csv', quiet=False)

df = pd.read_csv('IMDB-Dataset.csv')
print(f'Shape: {df.shape}')
print(df['sentiment'].value_counts())
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Removed {duplicates} duplicates. New shape: {df.shape}')
df.head()

---
## 3. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

print('Preprocessing...')
df['cleaned_review'] = df['review'].apply(preprocess_text)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print('Done.')

---
## 4. Train-Test Split (80:20)

In [ ]:
X = df['cleaned_review']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

---
## 5. Feature Extraction

### 5a. Bag of Words (300 features)

In [ ]:
bow_vectorizer = CountVectorizer(max_features=300)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)
print(f'BoW shape: {X_train_bow.shape}')

### 5b. TF-IDF (300 features)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=300, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print(f'TF-IDF shape: {X_train_tfidf.shape}')

---
## 6. Model Training & Evaluation

### 6a. SVM + BoW

In [ ]:
svm_bow = LinearSVC(max_iter=2000, random_state=42, C=1.0)
svm_bow.fit(X_train_bow, y_train)
y_pred_bow = svm_bow.predict(X_test_bow)

acc_bow  = accuracy_score(y_test, y_pred_bow)
prec_bow = precision_score(y_test, y_pred_bow)
rec_bow  = recall_score(y_test, y_pred_bow)
f1_bow   = f1_score(y_test, y_pred_bow)

print('=' * 55)
print('  SVM + BoW — RESULTS')
print('=' * 55)
print(f'  Accuracy:  {acc_bow:.4f}')
print(f'  Precision: {prec_bow:.4f}')
print(f'  Recall:    {rec_bow:.4f}')
print(f'  F1-Score:  {f1_bow:.4f}')
print('=' * 55)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_bow, target_names=['Negative', 'Positive']))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_bow)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive']).plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Confusion Matrix — SVM + BoW', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 6b. SVM + TF-IDF

In [ ]:
svm_tfidf = LinearSVC(max_iter=2000, random_state=42, C=1.0)
svm_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = svm_tfidf.predict(X_test_tfidf)

acc_tfidf  = accuracy_score(y_test, y_pred_tfidf)
prec_tfidf = precision_score(y_test, y_pred_tfidf)
rec_tfidf  = recall_score(y_test, y_pred_tfidf)
f1_tfidf   = f1_score(y_test, y_pred_tfidf)

print('=' * 55)
print('  SVM + TF-IDF — RESULTS')
print('=' * 55)
print(f'  Accuracy:  {acc_tfidf:.4f}')
print(f'  Precision: {prec_tfidf:.4f}')
print(f'  Recall:    {rec_tfidf:.4f}')
print(f'  F1-Score:  {f1_tfidf:.4f}')
print('=' * 55)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_tfidf, target_names=['Negative', 'Positive']))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_tfidf)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive']).plot(cmap='Oranges', ax=ax, values_format='d')
ax.set_title('Confusion Matrix — SVM + TF-IDF', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7. Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Model': ['SVM + BoW', 'SVM + TF-IDF'],
    'Accuracy': [acc_bow, acc_tfidf],
    'Precision': [prec_bow, prec_tfidf],
    'Recall': [rec_bow, rec_tfidf],
    'F1-Score': [f1_bow, f1_tfidf]
})
print(comparison.to_string(index=False))

---
## 8. Analysis and Discussion

### Observations
- **TF-IDF** typically yields better results than raw **BoW** due to discriminative term weighting.
- **SVM**'s maximum-margin principle is well-suited for high-dimensional sparse text features.
- Bigrams in TF-IDF help capture short sentiment-bearing phrases.

### Limitations
- Both approaches remain bag-of-words — no semantic or contextual understanding.
- 300 features is restrictive; higher dimensionality could improve performance.
- Cannot model word order or long-range dependencies.